# DYF Quickstart — Density Yields Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdonaldson/dyf/blob/main/docs/notebooks/quickstart.ipynb)

DYF discovers structure in embedding spaces using density-based LSH.
Every item gets classified as **Dense** (core), **Bridge** (connector), or **Orphan** (unique) — no k to pick, no hyperparameters to tune.

This notebook walks through:
1. Embedding real text data (AG News headlines)
2. Density classification — what's core, what's bridging, what's unique?
3. Building a `.dyf` search index
4. Searching with adaptive probing

## 1. Install dependencies

In [1]:
!pip install -q dyf[io] datasets sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 14.4 MB/s eta 0:00:00


## 2. Load AG News

AG News has 4 categories: World, Sports, Business, Sci/Tech. We grab 5,000 headlines for speed.

In [2]:
from datasets import load_dataset

ds = load_dataset("ag_news", split="train[:5000]")

AG_LABELS = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
texts = ds["text"]
categories = [AG_LABELS[l] for l in ds["label"]]

print(f"{len(texts)} headlines loaded")
print(f"Example: {texts[0][:100]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

5000 headlines loaded
Example: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...


## 3. Embed with sentence-transformers

`all-MiniLM-L6-v2` produces 384-d embeddings and runs fast on CPU.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

print(f"Embeddings shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


## 4. Density classification

DYF's `DensityClassifier` buckets items via PCA-based LSH, then computes per-item density metrics.
No k to choose, no cluster count to guess.

In [4]:
from dyf import DensityClassifier

classifier = DensityClassifier(
    embedding_dim=384,
    num_bits=14,
    num_stability_seeds=3,
)
classifier.fit(embeddings)

print(classifier.report())

Corpus: 5000 items
  Buckets: 2650
  Mean bucket size: 4.0
  Median bucket size: 2
  Max bucket size: 37
  Mean centroid similarity: 0.850
  Mean isolation score: 0.385
  Mean stability score: 0.001
  PCA variance explained: 100.0%


## 5. Inspect Dense, Bridge, and Orphan items

- **Dense**: High centroid similarity — the item sits squarely in a well-populated region
- **Bridge**: Low centroid similarity — the item connects different semantic regions
- **Orphan**: Small bucket — the item has few semantic neighbors

We use `analyze_bridges()` to find bridge items and their connections.

In [5]:
# Get per-item metrics (returned as lists, convert to arrays)
centroid_sims = np.array(classifier.get_centroid_similarities())
bucket_sizes = np.array(classifier.get_bucket_sizes())

# Simple classification from raw metrics
is_orphan = bucket_sizes <= 2
is_bridge = (~is_orphan) & (centroid_sims < 0.5)
is_dense = (~is_orphan) & (centroid_sims >= 0.5)

n = len(texts)
print(f"Dense:  {is_dense.sum():>5} ({is_dense.sum()/n:.1%}) — core items")
print(f"Bridge: {is_bridge.sum():>5} ({is_bridge.sum()/n:.1%}) — connectors")
print(f"Orphan: {is_orphan.sum():>5} ({is_orphan.sum()/n:.1%}) — unique items")

print("\n--- Example Dense headlines (high centroid similarity) ---")
dense_idx = np.where(is_dense)[0]
for i in dense_idx[:3]:
    print(f"  [{categories[i]}] sim={centroid_sims[i]:.3f}  {texts[i][:90]}")

print("\n--- Example Orphan headlines (tiny buckets) ---")
orphan_idx = np.where(is_orphan)[0]
for i in orphan_idx[:3]:
    print(f"  [{categories[i]}] bucket_size={bucket_sizes[i]}  {texts[i][:90]}")

Dense:   2257 (45.1%) — core items
Bridge:    87 (1.7%) — connectors
Orphan:  2656 (53.1%) — unique items

--- Example Dense headlines (high centroid similarity) ---


TypeError: Wrong key type: '1' of type '<class 'numpy.int64'>'. Expected one of int, slice, range, str or Iterable.

In [ ]:
# Bridge analysis — which regions do bridges connect?
analysis = classifier.analyze_bridges(embeddings, bridge_threshold=0.5)
print(analysis)

print("\nTop connected bucket pairs:")
for bucket_a, bucket_b, count in analysis.top_connected_pairs(5):
    print(f"  Bucket {bucket_a} <-> Bucket {bucket_b}: {count} bridges")

# Show a few bridge headlines
print("\n--- Example Bridge headlines ---")
for i in analysis.bridge_indices[:5]:
    print(f"  [{categories[i]}] sim={centroid_sims[i]:.3f}  {texts[i][:90]}")

## 6. Build a .dyf search index

A `.dyf` file is a memory-mapped index with zero startup cost. We store the headline text and category alongside the embeddings.

In [ ]:
from dyf import build_dyf_tree, write_lazy_index

tree = build_dyf_tree(embeddings, max_depth=4, num_bits=3, min_leaf_size=8)

write_lazy_index(
    tree, embeddings, "ag_news.dyf",
    quantization="float16",
    stored_fields={"title": texts, "category": categories},
    metadata={"model": "all-MiniLM-L6-v2", "dataset": "ag_news"},
)

print("Index written to ag_news.dyf")

## 7. Search the index

Open the `.dyf` file and search with a query embedding. Stored fields come back with results.

In [ ]:
from dyf import LazyIndex

query = model.encode("stock market crash")

with LazyIndex("ag_news.dyf") as idx:
    print(f"Index: {idx.total_items} items, {idx.embedding_dim}d embeddings")
    print(f"Stored fields: {idx.stored_field_names}\n")

    result = idx.search(query, k=10)

    print('Query: "stock market crash"\n')
    for i, (score, title, cat) in enumerate(
        zip(result.scores, result.fields["title"], result.fields["category"])
    ):
        print(f"  {i+1:2d}. [{cat}] {score:.3f}  {title[:80]}")

## 8. Adaptive probing

With `nprobe="auto"`, DYF decides how many tree leaves to search based on how confident the routing is.
Unambiguous queries probe fewer leaves (faster); ambiguous queries probe more (better recall).

In [ ]:
queries = [
    "NBA playoffs basketball",          # clearly Sports
    "new iPhone release",               # clearly Sci/Tech
    "tech companies report earnings",   # Sci/Tech + Business overlap
]

with LazyIndex("ag_news.dyf") as idx:
    for q in queries:
        qvec = model.encode(q)
        result = idx.search(qvec, k=5, nprobe="auto", return_routing=True)

        r = result.routing
        print(f'Query: "{q}"')
        print(f"  Leaves probed: {len(r['leaves_probed'])}  "
              f"Candidates scored: {r['candidates_scored']}  "
              f"Min margin: {r['min_margin']:.4f}  "
              f"Elapsed: {r['elapsed_ms']:.1f}ms")
        print(f"  Top result: [{result.fields['category'][0]}] "
              f"{result.fields['title'][0][:70]}")
        print()

## Next steps

- **[How It Works](https://dyf.io/how-it-works.html)** — the PCA-LSH algorithm, density metrics, and Krapivin hash density
- **[API Reference](https://dyf.io/reference/)** — full docs for `DensityClassifier`, `LazyIndex`, and more
- **[GitHub](https://github.com/jdonaldson/dyf)** — source, issues, and examples